# Gemma 3 1B: original 300,000-pair LoRA GPU recipe

The original L4 run trained Gemma 3 1B-IT for one pass over 300,000 official translation pairs.

## Read the GPU training record on CPU

**Run all** reads the saved training record and displays the historical checkpoint losses. Repeating the full GPU run requires the original clean-v2 inputs and authorized Gemma weights.

The optional cells preserve the original executed experiment's Torch/PEFT implementation: exact native chat boundaries, English-only causal loss, token-weighted gradient accumulation, length-aware batches, durable resume slots, objective/gradient checks and exact 10k/50k/300k adapters. This notebook adapts the original hosted notebook for a public clone. Evaluation is separate; validation loss alone does not establish translation quality.


In [1]:
from pathlib import Path
import json, subprocess, sys

# Locate a clone from repository root or notebooks/. A fresh Colab starts here too.
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "slavic-english-slm"]
ROOT = next((p for p in candidates if (p / "results/norm64_run.json").is_file()), None)
if ROOT is None:
    ROOT = Path.cwd() / "slavic-english-slm"
    if ROOT.exists():
        raise RuntimeError("Existing slavic-english-slm directory lacks the repository marker.")
    # The new notebook has no previously published pinned revision: bootstrap public main.
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "main",
            "https://github.com/Silveroboros-dev/slavic-english-slm.git",
            str(ROOT),
        ],
        check=True,
    )
assert (ROOT / "results/gemma_run.json").is_file()
print("Public repository located; CPU receipt replay ready.")
INSTALL_TRAINING_DEPENDENCIES = False  # Only consulted when RUN_TRAINING is True.
RUN_TRAINING = False  # Explicit opt-in only; a GPU rerun is conditional on the inputs below.
DATA_ROOT = None  # Path to your independently obtained, exact clean-v2 release directory.
MODEL_ROOT = None  # Path to your authorized official Gemma 3 1B-IT Transformers checkpoint.
OUTPUT_ROOT = None  # New writable persistent output directory; never the model directory.

Public repository located; CPU receipt replay ready.


In [2]:
receipt = json.loads((ROOT / "results/gemma_run.json").read_text())
assert receipt["data_sha256"] == "5b7a813459bf6129c08129ddf749ec5dc3655ad36225d55318b6cccc13b09429"
assert sum(receipt["pairs_by_language"].values()) == 300_000
assert receipt["optimizer_updates"] == 9376
print("ARCHIVE REPLAY — historical GPU metrics; no training performed in this execution")
print("Historical device:", receipt["hardware"])
print("Historical updates:", receipt["optimizer_updates"])
print("Historical timed training seconds:", round(receipt["training_seconds"], 3))
print("Historical trainable parameters:", receipt["trainable_parameters"])
for seen, loss in receipt["validation_mean_english_token_loss_by_examples_seen"].items():
    print(f"Historical validation English-token loss at {int(seen):,} examples: {loss:.6f}")
print("Timer excludes setup, validation, checkpoint writes, translation and scoring.")

ARCHIVE REPLAY — historical GPU metrics; no training performed in this execution
Historical device: NVIDIA L4
Historical updates: 9376
Historical timed training seconds: 7889.303
Historical trainable parameters: 745472
Historical validation English-token loss at 0 examples: 3.953490
Historical validation English-token loss at 10,000 examples: 1.215305
Historical validation English-token loss at 50,000 examples: 1.096388
Historical validation English-token loss at 300,000 examples: 0.950314
Timer excludes setup, validation, checkpoint writes, translation and scoring.


## Optional GPU run with the original data and weights

The default path above is usable immediately. Exact historical retraining is **conditional**: this public repository does not bundle the full clean-v2 300k file or model weights. Do not substitute another release and describe it as the same experiment. Obtain data lawfully and set `DATA_ROOT` to a directory containing `train.jsonl.gz`, `eval_inputs.json`, and `manifest.json`; the next cell verifies both exact file hashes. Rebuilding data is a separate task and a rebuilt file is accepted here only if its bytes match.

Use your own authorized official `google/gemma-3/transformers/gemma-3-1b-it/1` checkpoint (historical Kaggle version ID 282742), including tokenizer and weight files. This notebook performs no login or weight download. Set `MODEL_ROOT` to its local directory and `OUTPUT_ROOT` to your persistent output directory. The optional path verifies model.safetensors, config.json and tokenizer.json against the original clean-v2 protocol hashes before importing the training stack. It also verifies the rendered chat template, training example order, parameter count and token statistics against that protocol.

Select a bfloat16-capable L4/A100/H100 GPU. The original environment recorded Torch 2.14.1+cu130, Transformers 4.57.1, PEFT 0.17.1, NumPy 2.4.6; see the receipt for details. In your GPU environment install compatible CUDA Torch and `transformers==4.57.1 peft==0.17.1 numpy==2.4.6 safetensors`. For an optional library install, set `INSTALL_TRAINING_DEPENDENCIES=True` along with `RUN_TRAINING=True`; the install cell pins the three historical project libraries. CUDA Torch remains supplied by your environment; no automatic Torch replacement occurs. Then change `RUN_TRAINING=True` above and Run all. A rerun costs GPU time and writes checkpoints; it is not performed for the saved outputs here.


In [3]:
if RUN_TRAINING:
    if any(value is None for value in (DATA_ROOT, MODEL_ROOT, OUTPUT_ROOT)):
        raise ValueError(
            "Set DATA_ROOT, MODEL_ROOT and OUTPUT_ROOT before opting into GPU training."
        )
    from pathlib import Path
    import hashlib
    import json

    data_root = Path(DATA_ROOT)
    TRAIN_PATH = data_root / "train.jsonl.gz"
    SPLIT_CHECKSUM = "5b7a813459bf6129c08129ddf749ec5dc3655ad36225d55318b6cccc13b09429"
    EVALUATION_CHECKSUM = "b8326f75f7ad914e9ea94d1ad294c33644a38c2c351d99d2906241a67b550ac5"

    def file_sha256(path):
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    data_manifest = json.loads((data_root / "manifest.json").read_text(encoding="utf-8"))
    assert data_manifest["output_sha256"]["train.jsonl.gz"] == SPLIT_CHECKSUM
    assert data_manifest["output_sha256"]["eval_inputs.json"] == EVALUATION_CHECKSUM
    assert file_sha256(TRAIN_PATH) == SPLIT_CHECKSUM
    eval_input = data_root / "eval_inputs.json"
    assert file_sha256(eval_input) == EVALUATION_CHECKSUM
    eval_sets = json.loads(eval_input.read_text(encoding="utf-8"))
    VAL_ROWS = eval_sets["validation"]
    assert len(VAL_ROWS) == 48 and data_manifest["training_rows"] == 300_000
    assert data_manifest["source_counts"] == {"pl": 100_000, "hr": 100_000, "ru": 100_000}
    print("Verified the full 300,000-pair official training file and four frozen reference panels.")

    assert data_manifest["version"] == "gemma3-1b-full-official-clean-v2"

else:
    print("GPU training cell skipped (RUN_TRAINING=False).")

GPU training cell skipped (RUN_TRAINING=False).


In [4]:
if RUN_TRAINING:
    # Model fingerprint from the archived clean-v2 run protocol, not an inferred model name.
    import hashlib

    expected_model_hashes = {
        "model.safetensors": "3d4ef8d71c14db7e448a09ebe891cfb6bf32c57a9b44499ae0d1c098e48516b6",
        "config.json": "4dd5218cebc4fdea9004b944d34672a2d47f9ef930e3bb16a3fc4bbbe0fcee81",
        "tokenizer.json": "7d4046bf0505a327dd5a0abbb427ecd4fc82f99c2ceaa170bc61ecde12809b0c",
    }
    model_root = Path(MODEL_ROOT)
    for name, expected in expected_model_hashes.items():
        path = model_root / name
        if not path.is_file():
            raise FileNotFoundError(f"Exact historical official checkpoint requires {name}")
        h = hashlib.sha256()
        with path.open("rb") as f:
            for block in iter(lambda: f.read(4 * 1024 * 1024), b""):
                h.update(block)
        if h.hexdigest() != expected:
            raise ValueError(f"Historical model snapshot mismatch: {name}")
    print("Verified original official model weight/config/tokenizer byte hashes.")
    if INSTALL_TRAINING_DEPENDENCIES:
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "transformers==4.57.1",
                "peft==0.17.1",
                "numpy==2.4.6",
                "safetensors",
            ],
            check=True,
        )
        print(
            "Project dependencies installed; restart the kernel if they were previously imported."
        )
else:
    print("Model verification and dependency installation skipped (RUN_TRAINING=False).")

Model verification and dependency installation skipped (RUN_TRAINING=False).


In [5]:
if RUN_TRAINING:
    import gc, gzip, hashlib, itertools, json, math, os, random, shutil, time, uuid
    from array import array
    from collections import Counter
    from importlib.metadata import version
    from pathlib import Path
    import numpy as np
    import torch
    import torch.nn.functional as F
    from torch.utils.checkpoint import checkpoint
    from peft import (
        LoraConfig,
        get_peft_model,
        get_peft_model_state_dict,
        set_peft_model_state_dict,
    )
    from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

    MODEL_ID = "google/gemma-3/transformers/gemma-3-1b-it/1"
    MODEL_REVISION = "Kaggle model version ID 282742"
    SEED = 42
    INSTRUCTION = (
        "Translate the following {language} message into English. Return only the translation."
    )
    LANGUAGES = {"pl": "Polish", "hr": "Croatian", "ru": "Russian", "uk": "Ukrainian"}
    RUN_DIR = Path(OUTPUT_ROOT)
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    if not torch.cuda.is_available() or not any(
        name in torch.cuda.get_device_name() for name in ["H100", "A100", "L4"]
    ):
        raise RuntimeError("Select one hosted A100, H100 or L4 GPU before running this experiment.")
    dtype = torch.bfloat16
    device = torch.device("cuda")
    print(
        torch.cuda.get_device_name(),
        torch.cuda.get_device_properties(0).total_memory // 2**30,
        "GiB",
    )
    local_model = Path(MODEL_ROOT)
    assert RUN_DIR.resolve() != local_model.resolve(), (
        "Outputs must not overwrite the base model directory."
    )
    if not (local_model / "config.json").is_file():
        candidates = [
            p.parent
            for p in local_model.rglob("config.json")
            if any(p.parent.glob("*.safetensors"))
        ]
        if len(candidates) != 1:
            raise ValueError(
                "The Kaggle artifact does not have a unique Transformers checkpoint directory."
            )
        local_model = candidates[0]
    tokenizer = AutoTokenizer.from_pretrained(
        local_model, local_files_only=True, trust_remote_code=False
    )
    template_source = "Kaggle tokenizer"
    if not tokenizer.chat_template:
        # Older exports may omit the Jinja metadata. This is Gemma's native text-only turn format.
        tokenizer.chat_template = (
            "{{ bos_token }}{% for m in messages %}"
            "{{ '<start_of_turn>' + ('model' if m.role == 'assistant' else 'user') + '\\n' "
            "+ (m.content | trim) + '<end_of_turn>\\n' }}{% endfor %}"
            "{% if add_generation_prompt %}{{ '<start_of_turn>model\\n' }}{% endif %}"
        )
        template_source = "Gemma text-only native turns"
    turn_end = tokenizer.convert_tokens_to_ids("<end_of_turn>")
    if turn_end == tokenizer.unk_token_id or tokenizer.pad_token_id is None:
        raise ValueError("Expected Gemma turn and padding tokens.")

    def load_base():
        return AutoModelForCausalLM.from_pretrained(
            local_model,
            torch_dtype=dtype,
            attn_implementation="sdpa",
            local_files_only=True,
            trust_remote_code=False,
        ).to(device)

    def digest(value):
        raw = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()
        return hashlib.sha256(raw).hexdigest()

    def atomic_json(path, data):
        temp = path.with_name(path.name + "." + uuid.uuid4().hex + ".part")
        temp.write_text(
            json.dumps(data, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8"
        )
        os.replace(temp, path)

    MODEL_DIR = local_model
    MILESTONES = (10_000, 50_000, 300_000)
    EFFECTIVE_BATCH = 32
    MICROBATCH = 8
    PADDED_TOKEN_BUDGET = 1_536

    def file_sha(path):
        value = hashlib.sha256()
        with open(path, "rb") as f:
            for block in iter(lambda: f.read(4_194_304), b""):
                value.update(block)
        return value.hexdigest()

else:
    print("GPU training cell skipped (RUN_TRAINING=False).")

GPU training cell skipped (RUN_TRAINING=False).


## Full chat examples and length-aware batches
Preserve every source, English target and end-of-turn token. Keep short and long examples in randomly ordered, length-matched batches. Flush a partial optimizer batch at the exact milestones.

In [6]:
if RUN_TRAINING:
    # Render the native chat template just once per language. The two slots are plain text.
    def render(lang, source, target=None):
        user = INSTRUCTION.format(language=LANGUAGES[lang]) + "\n\n" + source
        turns = [{"role": "user", "content": user}]
        if target is not None:
            turns.append({"role": "assistant", "content": target})
        return tokenizer.apply_chat_template(
            turns, tokenize=False, add_generation_prompt=target is None
        )

    source_slot, answer_slot = "GEMMASOURCE87124", "GEMMAANSWER87124"
    parts = {}
    for lang in ("hr", "pl", "ru"):
        prefix, middle = render(lang, source_slot).split(source_slot)
        whole = render(lang, source_slot, answer_slot)
        ending = whole.split(answer_slot)[1]
        assert whole == prefix + source_slot + middle + answer_slot + ending
        assert ending.startswith("<end_of_turn>")
        parts[lang] = (prefix, middle)
    special_tokens = tuple(tokenizer.all_special_tokens)

    def encode_rows(rows):
        prompts, completed = [], []
        for row in rows:
            src, tgt = row["src"].strip(), row["tgt"].strip()
            # Match the reference chat_template, which trims message boundaries.
            assert src and tgt and not any(t in src or t in tgt for t in special_tokens), row["id"]
            start, role = parts[row["lang"]]
            prompt = start + src + role
            prompts.append(prompt)
            completed.append(prompt + tgt + "<end_of_turn>")
        ids = tokenizer(
            prompts + completed,
            add_special_tokens=False,
            truncation=False,
            return_attention_mask=False,
            verbose=False,
        )["input_ids"]
        for row, p, c in zip(rows, ids[: len(rows)], ids[len(rows) :]):
            if c[: len(p)] != p or c[-1] != turn_end or len(c) <= len(p) + 1:
                raise ValueError(f"Unexpected target boundary in {row['id']}")
            yield c, len(p)

    def encoded_table(row_iter):
        packed = array("I")
        offsets, prompts = [0], []
        it = iter(row_iter)
        while rows := list(itertools.islice(it, 512)):
            for ids, prompt_len in encode_rows(rows):
                packed.extend(ids)
                offsets.append(len(packed))
                prompts.append(prompt_len)
        return (
            np.frombuffer(packed, np.uint32),
            np.array(offsets, dtype=np.int64),
            np.array(prompts, dtype=np.int32),
        )

    # Compare full native Jinja output with slot-based tokenization on two actual rows per language.
    for lang in parts:
        examples = [r for r in VAL_ROWS if r["lang"] == lang][:2]
        assert len(examples) == 2
        for row, (packed_ids, prefix_len) in zip(examples, encode_rows(examples)):
            canonical = render(lang, row["src"].strip(), row["tgt"].strip()).rstrip()
            check = tokenizer(canonical, add_special_tokens=False)["input_ids"]
            assert (
                check == packed_ids
                and len(
                    tokenizer(render(lang, row["src"].strip()), add_special_tokens=False)[
                        "input_ids"
                    ]
                )
                == prefix_len
            )
    print("Verified native chat construction in all three training languages.", flush=True)

    with gzip.open(TRAIN_PATH, "rt", encoding="utf-8") as f:
        train = encoded_table(map(json.loads, f))
    val = encoded_table(VAL_ROWS)
    lengths = np.diff(train[1])
    assert len(lengths) == 300_000

    # Restore the very same order after interruption. Shuffle examples first so milestones are a random slice.
    # Inside windows, take equal-length groups and shuffle the groups to avoid a short-to-long curriculum.
    def make_batches(lengths, milestones):
        rng = np.random.default_rng(SEED)
        shuffled = rng.permutation(len(lengths))
        batches, start = [], 0
        for stop in milestones:
            for i in range(start, stop, 1024):
                window = shuffled[i : min(i + 1024, stop)]
                window = window[np.argsort(lengths[window], kind="stable")]
                groups = [
                    window[j : j + EFFECTIVE_BATCH] for j in range(0, len(window), EFFECTIVE_BATCH)
                ]
                # Keep the partial batch last so progress counts remain interpretable.
                full = len(groups) - int(len(groups[-1]) != EFFECTIVE_BATCH)
                batches += [groups[k] for k in rng.permutation(full)]
                batches += groups[full:]
            start = stop
        return batches

    batches = make_batches(lengths, MILESTONES)
    order = np.concatenate(batches)
    end_positions = np.cumsum([len(g) for g in batches])
    assert np.array_equal(np.sort(order), np.arange(300_000))
    assert set(MILESTONES) <= set(end_positions)
    print(
        f"{len(batches):,} updates. Longest native sequence: {int(lengths.max()):,} tokens; no truncation."
    )
    print("Median and 99th percentile:", np.percentile(lengths, [50, 99]).tolist())

    print(
        "Longest row is seen by optimizer step",
        next(i for i, g in enumerate(batches, 1) if int(lengths.argmax()) in g),
    )

else:
    print("GPU training cell skipped (RUN_TRAINING=False).")

GPU training cell skipped (RUN_TRAINING=False).


## English-target objective
Right-pad each microbatch and accumulate the summed English cross-entropy against the number of English tokens in its effective batch. The native output projection is needed only at supervised positions. Extremely long single records use activation recomputation.

In [7]:
if RUN_TRAINING:
    set_seed(SEED)
    base = load_base()
    assert lengths.max() <= base.config.max_position_embeddings
    lora_cfg = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.0,
        bias="none",
        target_modules=["q_proj", "v_proj"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base, lora_cfg)
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=2e-4, weight_decay=0.0, fused=True)
    model.print_trainable_parameters()
    # Normal microbatches keep their activations; unusually long records trigger non-reentrant recomputation.
    native = model.get_base_model()
    native.config.use_cache = False

    def microgroups(table, rows, max_rows):
        sequence_lengths = table[1][rows + 1] - table[1][rows]
        start = 0
        while start < len(rows):
            stop = min(start + max_rows, len(rows))
            while (
                stop > start + 1
                and (stop - start) * int(max(sequence_lengths[start:stop])) > PADDED_TOKEN_BUDGET
            ):
                stop -= 1
            yield rows[start:stop]
            start = stop

    def pad_inputs(table, rows):
        packed, offsets, prefix_lengths = table
        sizes = offsets[rows + 1] - offsets[rows]
        width = (int(sizes.max()) + 7) // 8 * 8
        x = np.full((len(rows), width), tokenizer.pad_token_id, dtype=np.int64)
        labels = np.full_like(x, -100)
        valid = np.zeros_like(x, dtype=np.bool_)
        for r, (idx, n) in enumerate(zip(rows, sizes)):
            ids = packed[offsets[idx] : offsets[idx + 1]]
            x[r, :n] = ids
            valid[r, :n] = True
            p = prefix_lengths[idx]
            labels[r, p:n] = ids[p:]
        return tuple(torch.from_numpy(a).to(device) for a in (x, valid, labels))

    def loss_sum(table, rows):
        ids, valid, labels = pad_inputs(table, rows)
        # HF's shifted causal loss with padding and source tokens masked. Only project the positions
        # that actually contribute to the English loss, including <end_of_turn>.
        checkpoint_blocks = ids.shape[1] > 1024 and model.training
        if checkpoint_blocks and not native.is_gradient_checkpointing:
            native.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={"use_reentrant": False}
            )
        elif not checkpoint_blocks and native.is_gradient_checkpointing:
            native.gradient_checkpointing_disable()
        hidden = native.model(
            input_ids=ids, attention_mask=valid, use_cache=False
        ).last_hidden_state
        selected = labels[:, 1:] != -100
        h = hidden[:, :-1][selected]
        targets = labels[:, 1:][selected]

        def project(block, y):
            logits = native.lm_head(block)
            cap = native.config.final_logit_softcapping
            if cap is not None:
                logits = (logits / cap).tanh() * cap
            return F.cross_entropy(logits.float(), y, reduction="sum")

        # Bounded peak logit memory even for a single exceptionally long target.
        if len(h) > 512 and model.training:
            return sum(
                checkpoint(project, block, y, use_reentrant=False)
                for block, y in zip(h.split(256), targets.split(256))
            )
        return sum(project(block, y) for block, y in zip(h.split(512), targets.split(512)))

    def target_count(table, rows):
        return int((table[1][rows + 1] - table[1][rows] - table[2][rows]).sum())

    def backward_batch(table, rows, max_rows=MICROBATCH):
        total = target_count(table, rows)
        loss = torch.zeros((), device=device)
        for group in microgroups(table, rows, max_rows):
            value = loss_sum(table, group)
            (value / total).backward()
            loss += value.detach()
        return loss / total

    @torch.inference_mode()
    def validation():
        model.eval()
        losses, nt = {}, {}
        for lang in sorted(parts):
            rows = np.array([i for i, r in enumerate(VAL_ROWS) if r["lang"] == lang])
            scores = [float(loss_sum(val, g)) for g in microgroups(val, rows, MICROBATCH)]
            nt[lang] = target_count(val, rows)
            losses[lang] = sum(scores) / nt[lang]
        model.train()
        return {
            "target_token_loss": sum(losses[k] * nt[k] for k in losses) / sum(nt.values()),
            "language_loss": losses,
            "supervised_tokens": nt,
            "rows": len(VAL_ROWS),
        }

    protocol = {
        "format": "gemma3-full-official-v2",
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "train_sha256": SPLIT_CHECKSUM,
        "validation_sha256": digest(VAL_ROWS),
        "order_sha256": hashlib.sha256(order.astype("<u4").tobytes()).hexdigest(),
        "order": "numpy default_rng, window 1024, groups of 32; flush at milestones",
        "steps": len(batches),
        "seed": SEED,
        "milestones": list(MILESTONES),
        "effective_batch": EFFECTIVE_BATCH,
        "microbatch_max": MICROBATCH,
        "max_padded_microbatch_tokens": PADDED_TOKEN_BUDGET,
        "single_long_records": "never truncated",
        "trainable_parameters": sum(p.numel() for p in trainable),
        "adamw_lr": 2e-4,
        "weight_decay": 0.0,
        "gradient_clip": 1.0,
        "scheduler": "constant",
        "rank": 8,
        "alpha": 16,
        "lora_modules": ["q_proj", "v_proj"],
        "lora_dropout": 0.0,
        "instruction": INSTRUCTION,
        "template_sha256": hashlib.sha256(tokenizer.chat_template.encode()).hexdigest(),
        "dtype": "bfloat16",
        "attention": "sdpa",
        "versions": {p: version(p) for p in ("torch", "transformers", "peft", "numpy")},
        "length": {
            "min": int(lengths.min()),
            "median": float(np.median(lengths)),
            "p99": float(np.percentile(lengths, 99)),
            "max": int(lengths.max()),
            "source_and_target_tokens": int(lengths.sum()),
            "english_tokens_including_eot": int((lengths - train[2]).sum()),
        },
        "local_file_sha256": {
            p.name: file_sha(p)
            for p in list(local_model.glob("*.safetensors"))
            + [local_model / "config.json", local_model / "tokenizer.json"]
        },
    }
    # Exact clean-v2 archival protocol cross-checks; a different model/data/tokenizer must fail.
    assert (
        protocol["validation_sha256"]
        == "b64b9c574ed6294bfb36a09a115ed2f82584a1c167162ff19f4090ebe0634113"
    )
    assert (
        protocol["order_sha256"]
        == "503c4f3909b62a5be7b97e747b9e5d50b97ace318648a1887e0976325f2953c5"
    )
    assert (
        protocol["template_sha256"]
        == "5437d0ad30bb8db69f45cd62c6e02d4269ee617ec9637c9650f8f04051bd9dbc"
    )
    assert protocol["steps"] == 9376 and protocol["trainable_parameters"] == 745472
    assert protocol["length"]["max"] == 2352
    assert protocol["length"]["source_and_target_tokens"] == 29955548
    assert protocol["length"]["english_tokens_including_eot"] == 9670521
    run_sig = digest(protocol)
    if (RUN_DIR / "protocol.json").is_file():
        assert digest(json.loads((RUN_DIR / "protocol.json").read_text())) == run_sig, (
            "Existing protocol differs."
        )
    else:
        atomic_json(RUN_DIR / "protocol.json", protocol)
    print("Training protocol", run_sig)

else:
    print("GPU training cell skipped (RUN_TRAINING=False).")

GPU training cell skipped (RUN_TRAINING=False).


## Recovery, validation and matched batching benchmark

Each durable checkpoint includes AdamW state, Torch RNG and example cursor. Before training, compare optimized loss and adapter gradients against the native Transformers objective at identical tensor shapes; check English-token counts and masked-padding invariance. Record single-row versus padded bfloat16 differences separately. The matched benchmark compares microbatch 1 and 8 on the same 32 rows without updating weights. These checks execute only in the optional GPU run.


In [8]:
if RUN_TRAINING:
    from safetensors.torch import load_file
    import datetime

    def now():
        return datetime.datetime.now(datetime.timezone.utc).isoformat()

    def persist_checkpoint(step, examples, slot, elapsed, validation_records):
        target = RUN_DIR / f"resume-{slot}.pt"
        temp = target.with_name(f".writing-{slot}.pt")
        snapshot = {
            "signature": run_sig,
            "step": step,
            "examples": examples,
            "training_seconds": elapsed,
            "device": torch.cuda.get_device_name(),
            "validation": validation_records,
            "adapter": {k: v.detach().cpu() for k, v in get_peft_model_state_dict(model).items()},
            "optimizer": optimizer.state_dict(),
            "cpu_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state(),
        }
        torch.save(snapshot, temp)
        signature = file_sha(temp)
        os.replace(temp, target)
        receipt = {
            "step": step,
            "examples": examples,
            "slot": slot,
            "sha256": signature,
            "bytes": target.stat().st_size,
            "training_seconds": elapsed,
            "saved_utc": now(),
            "device": torch.cuda.get_device_name(),
        }
        atomic_json(RUN_DIR / f"resume-{slot}.json", receipt)
        atomic_json(RUN_DIR / "progress.json", receipt)

    def resume():
        candidates = []
        for slot in ("a", "b"):
            path = RUN_DIR / f"resume-{slot}.json"
            if path.is_file():
                candidates.append(json.loads(path.read_text()))
        for manifest in sorted(candidates, key=lambda p: p["step"], reverse=True):
            slot = manifest["slot"]
            path = RUN_DIR / f"resume-{slot}.pt"
            if not path.is_file() or file_sha(path) != manifest["sha256"]:
                continue
            snap = torch.load(path, map_location="cpu", weights_only=True)
            assert (
                snap["signature"] == run_sig
                and manifest["step"] == snap["step"]
                and snap["examples"] == int(end_positions[snap["step"] - 1])
            )
            set_peft_model_state_dict(model, snap["adapter"])
            optimizer.load_state_dict(snap["optimizer"])
            torch.set_rng_state(snap["cpu_rng"])
            torch.cuda.set_rng_state(snap["cuda_rng"])
            return snap["step"], snap["training_seconds"], snap["validation"], slot
        if candidates:
            raise ValueError("Neither persisted checkpoint passed checksum verification.")
        return 0, 0.0, {}, None

    def publish_adapter(step, seen, validations):
        dest = RUN_DIR / f"adapter-{seen}"
        dest.mkdir(exist_ok=True)
        marker = dest / "complete.json"
        if marker.exists():
            m = json.loads(marker.read_text())
            assert m["protocol_sha256"] == run_sig and m["adapter_sha256"] == file_sha(
                dest / "adapter_model.safetensors"
            )
            assert (
                m["config_sha256"] == file_sha(dest / "adapter_config.json")
                and m["examples_seen"] == seen
            )
            return
        model.save_pretrained(dest, safe_serialization=True, save_embedding_layers=False)
        weights = load_file(str(dest / "adapter_model.safetensors"))
        current = get_peft_model_state_dict(model)
        assert set(weights) == set(current) and all(
            torch.equal(weights[k], current[k].detach().cpu()) for k in weights
        )
        # Verify predictions after fully reloading the saved adapter onto the same immutable backbone.
        keys = set_peft_model_state_dict(model, weights)
        assert not keys.unexpected_keys
        new_loss = validation()["target_token_loss"]
        old_loss = validations[str(seen)]["target_token_loss"]
        assert abs(old_loss - new_loss) < 1e-5, (old_loss, new_loss)
        atomic_json(
            marker,
            {
                "global_step": step,
                "examples_seen": seen,
                "protocol_sha256": run_sig,
                "adapter_sha256": file_sha(dest / "adapter_model.safetensors"),
                "config_sha256": file_sha(dest / "adapter_config.json"),
                "reload_validation_loss": new_loss,
                "saved_utc": now(),
            },
        )

    step, accumulated, validation_history, slot = resume()
    if "0" not in validation_history:
        validation_history["0"] = validation()
    if step and int(end_positions[step - 1]) in MILESTONES:
        publish_adapter(step, int(end_positions[step - 1]), validation_history)
    atomic_json(RUN_DIR / "validation-loss.json", validation_history)

    # Check loss, one adapter gradient and single-vs-padded target weighting across all three languages.
    idx = np.array(
        [next(i for i, r in enumerate(VAL_ROWS) if r["lang"] == lang) for lang in sorted(parts)]
    )
    x, mask, y = pad_inputs(val, idx)
    model.eval()
    optimizer.zero_grad(set_to_none=True)
    usual = native(input_ids=x, attention_mask=mask, labels=y, use_cache=False).loss
    usual.backward()
    check_key, check_param = next(
        (k, p)
        for k, p in model.named_parameters()
        if "lora_B" in k and p.grad is not None and p.grad.norm().item() > 0
    )
    ref_grad = check_param.grad.detach().clone()
    optimizer.zero_grad(set_to_none=True)
    optimized = loss_sum(val, idx) / target_count(val, idx)
    optimized.backward()
    candidate_grad = check_param.grad.detach()
    cosine = float(F.cosine_similarity(ref_grad.flatten(), candidate_grad.flatten(), dim=0))
    assert torch.isfinite(candidate_grad).all() and cosine > 0.995
    with torch.no_grad():
        usual_padded = float(usual.detach())
        single_sum = 0.0
        max_single_objective_gap = 0.0
        for row in idx:
            one = np.array([row])
            xx, mm, yy = pad_inputs(val, one)
            ref = float(native(input_ids=xx, attention_mask=mm, labels=yy, use_cache=False).loss)
            count = target_count(val, one)
            single_score = float(loss_sum(val, one)) / count
            max_single_objective_gap = max(max_single_objective_gap, abs(single_score - ref))
            single_sum += count * ref
        single_weighted = single_sum / target_count(val, idx)
        # Hold tensor shapes fixed while perturbing every padding-token ID. Their indices are masked.
        assert torch.equal(y[~mask], torch.full_like(y[~mask], -100))
        assert int((y != -100).sum()) == target_count(val, idx)
        changed_padding = x.masked_fill(~mask, turn_end)
        perturbed = float(
            native(input_ids=changed_padding, attention_mask=mask, labels=y, use_cache=False).loss
        )
    assert abs(usual_padded - float(optimized.detach())) < 0.0001
    assert max_single_objective_gap < 0.0001 and abs(usual_padded - perturbed) < 0.0001
    atomic_json(
        RUN_DIR / "objective-check.json",
        {
            "native_hf_loss": usual_padded,
            "target_only_loss": float(optimized.detach()),
            "same_shape_padding_perturbation_loss": perturbed,
            "single_example_objective_max_absolute_gap": max_single_objective_gap,
            "native_weighted_single_rows_loss": single_weighted,
            "native_bf16_batching_difference": usual_padded - single_weighted,
            "checked_lora_tensor": check_key,
            "gradient_cosine": cosine,
            "examples": len(idx),
        },
    )
    optimizer.zero_grad(set_to_none=True)
    del x, mask, y, usual, optimized, ref_grad, candidate_grad
    model.train()

    bench_path = RUN_DIR / "batching-benchmark.json"
    if not bench_path.exists() and step == 0:
        matched = np.random.default_rng(SEED + 1).choice(len(lengths), size=32, replace=False)
        matched = matched[np.argsort(lengths[matched], kind="stable")]
        benchmark = []
        # Warm up each method. Repeat both in reverse order to limit first-method and clock bias.
        for cap in (1, MICROBATCH, 1, MICROBATCH, MICROBATCH, 1):
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            begin = time.monotonic()
            loss = float(backward_batch(train, matched, cap))
            torch.cuda.synchronize()
            seconds = time.monotonic() - begin
            benchmark.append(
                {
                    "microbatch_limit": cap,
                    "seconds": seconds,
                    "loss": loss,
                    "examples_per_second": len(matched) / seconds,
                    "peak_allocated_GiB": torch.cuda.max_memory_allocated() / 2**30,
                }
            )
        optimizer.zero_grad(set_to_none=True)
        assert all(math.isfinite(b["loss"]) and b["seconds"] > 0 for b in benchmark)
        atomic_json(
            bench_path,
            {
                "device": torch.cuda.get_device_name(),
                "rows": len(matched),
                "target_tokens": target_count(train, matched),
                "seed": SEED + 1,
                "sample_order_sha256": hashlib.sha256(matched.astype("<u4").tobytes()).hexdigest(),
                "measurement": "Forward and backward only, identical weights and sorted examples, no optimizer update. Two warmups excluded.",
                "runs": benchmark[2:],
            },
        )
        print("Matched batching benchmark", benchmark[2:])

else:
    print("GPU training cell skipped (RUN_TRAINING=False).")

GPU training cell skipped (RUN_TRAINING=False).


## Optional training: one epoch and exact milestone adapters

Checkpoint slots include optimizer, adapter, Torch RNG, cursor, protocol signature and checksum. Milestone adapters are loaded back into the live model and their validation loss checked. Keep the output directory persistent yourself; this public notebook does not mount private storage.


In [9]:
if RUN_TRAINING:
    session_started = now()
    session_steps = 0
    rolling_loss = 0.0
    rolling_count = 0
    model.train()
    for i in range(step, len(batches)):
        rows = batches[i]
        begin = time.monotonic()
        optimizer.zero_grad(set_to_none=True)
        batch_loss = backward_batch(train, rows)
        torch.nn.utils.clip_grad_norm_(trainable, 1.0, error_if_nonfinite=True)
        optimizer.step()
        loss = float(batch_loss)
        torch.cuda.synchronize()
        accumulated += time.monotonic() - begin
        rolling_loss += loss
        rolling_count += 1
        step = i + 1
        seen = int(end_positions[i])
        session_steps += 1
        if step <= 1 or step % 100 == 0 or seen in MILESTONES:
            print(
                f"Step {step:,}/{len(batches):,}; {seen:,} examples; "
                f"recent mean batch loss {rolling_loss / rolling_count:.4f}; {seen / accumulated:.1f} examples/sec.",
                flush=True,
            )
            rolling_loss, rolling_count = 0.0, 0
        if step == 100:
            atomic_json(
                RUN_DIR / "first-100-steps.json",
                {
                    "steps": step,
                    "seen": seen,
                    "training_seconds": accumulated,
                    "examples_per_second": seen / accumulated,
                    "seconds_per_step": accumulated / step,
                    "device": torch.cuda.get_device_name(),
                    "excludes": "Preparation, validation and persistence; includes padding, CPU-to-device, forward, backward and optimizer.",
                },
            )
        if seen in MILESTONES:
            validation_history[str(seen)] = validation()
        if step in (1, 100) or step % 250 == 0 or seen in MILESTONES:
            slot = "b" if slot == "a" else "a"
            persist_checkpoint(step, seen, slot, accumulated, validation_history)
            if seen in MILESTONES:
                publish_adapter(step, seen, validation_history)
                atomic_json(RUN_DIR / "validation-loss.json", validation_history)
                print("Verified milestone", seen, validation_history[str(seen)], flush=True)
    assert (RUN_DIR / "adapter-300000" / "complete.json").is_file()
    print("All three adapters persisted; training finished.")
    del native, base, trainable, optimizer, model
    gc.collect()
    torch.cuda.empty_cache()

else:
    print("GPU training cell skipped (RUN_TRAINING=False).")

GPU training cell skipped (RUN_TRAINING=False).


## How to interpret the training and validation losses

A successful optional rerun writes `protocol.json`, alternating `resume-a/b.pt` and checksum receipts, `validation-loss.json`, `objective-check.json`, `batching-benchmark.json`, and adapters at 10,000, 50,000 and 300,000 examples. Only load resume files you created and trust. The protocol signature prevents a changed run from silently resuming.

Historical validation mean loss fell from 3.953490 to 0.950314 after 300k examples; this is a saved result, not a CPU measurement. The implementation uses all source/target tokens without truncation; source and padding labels are masked, while English text and end-of-turn tokens are supervised. Optimized projection is checked against native Transformers loss and adapter gradients before training. Dynamic microbatching keeps at most eight rows and a 1,536-token padded budget where possible; a single long record is allowed beyond that budget, using activation checkpointing. Effective batches contain up to 32 examples and flush at exact milestones. The loss normalization uses English-token counts across the entire effective batch.

No model weights, credentials, private download links or private storage dependencies are included. The saved outputs below demonstrate only the public CPU receipt replay and deliberate skipping of GPU work. Use the repository's evaluation notebook/results to inspect held-out translation metrics separately.
